# Ensemble MLP Tutorial

Three uncertainty-aware surrogate modes on GFP brightness prediction:

- **Seed ensemble** — N independently-seeded networks
- **MC dropout** — one network, T stochastic inference passes
- **Combined** — N networks × T passes

Each section trains one variant, predicts, and plots. Section 6 compares all three.

In [ ]:
import time
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import spearmanr
import torch

from alf_core import BaseDatasetConfig, Candidate, LabelledCandidates, Modality
from alf_tools.datasets.gfp import GFP
from alf_tools.models.mlp import MLPModel, MLPModelConfig, MLPTrainConfig
from alf_tools.models.ensemble import EnsembleWrapper, EnsembleWrapperConfig

logging.basicConfig(level=logging.INFO, format="%(message)s")

## 1. Setup

In [ ]:
DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
FAST_MODE   = DEVICE == "cpu"
N_MEMBERS   = 10
N_EPOCHS    = 25 if FAST_MODE else 50
N_MC_PASSES = 10 if FAST_MODE else 20
BASE_SEED   = 42
DROPOUT_P   = 0.1
HIDDEN_DIMS = [128, 64]

print(f"Device : {DEVICE}")
print(f"Mode   : {'FAST (CPU)' if FAST_MODE else 'FULL (GPU)'}")
print(f"N_MEMBERS={N_MEMBERS}  N_EPOCHS={N_EPOCHS}  N_MC_PASSES={N_MC_PASSES}")